# TT/MPS基礎 11 — TT-SVD の準最適性と誤差上界

## 今回の位置づけ

Notebook 09 では、mixed-canonical TT/MPS の **単一 bond truncation** を扱いました。

Notebook 10 では、Eckart–Young–Mirsky（EYM）定理により、

$$
\operatorname{rank}(B)\le k
$$

を満たす任意の行列 $B$ に対して、truncated SVD

$$
A_k
=
U_k\Sigma_kV_k^T
$$

が Frobenius ノルムで最良の rank-$k$ 近似になることを確認しました。

つまり、1つの行列・1つの bond については、

$$
\boxed{
\|A-A_k\|_F
=
\min_{\operatorname{rank}(B)\le k}
\|A-B\|_F
}
$$

です。

今回は、これを **密な高階テンソルから左→右に TT を構成する標準 TT-SVD** へ広げます。

複数の cut で順に SVD truncation を行うため、単一 bond の EYM だけでは「最終 TT が大域最適」とは言えません。

その代わり、TT-SVD には

$$
\boxed{
\|A-T\|_F
\le
\left(
\sum_{k=1}^{d-1}
\varepsilon_k^2
\right)^{1/2}
}
$$

という誤差上界があります。

さらに、指定 TT-rank 以下での大域最適近似を $A_\ast$ とすると、

$$
\boxed{
\|A-T\|_F
\le
\sqrt{d-1}
\,
\|A-A_\ast\|_F
}
$$

となります。

この意味で、TT-SVD は **大域最適とは限らないが、最良 TT 近似から一定係数以内に保証される準最適法**です。

### 今回やること

1. 単一行列の SVD truncation を、複数特異値を捨てる例で再確認する
2. 標準 TT-SVD の左→右構成を整理する
3. rank 制限なしの TT-SVD で完全再構成を確認する
4. 固定 TT-rank の TT-SVD を実装する
5. 元テンソルの各 unfolding から $\varepsilon_k$ を計算する
6. 実測誤差と
   $$
   \sqrt{\sum_k\varepsilon_k^2}
   $$
   を比較する
7. 列直交行列 $U$ について、
   $$
   \|UZ\|_F=\|Z\|_F,
   \qquad
   \|U^TY\|_F\le\|Y\|_F
   $$
   を数値確認する

### 今回はまだ扱わないもの

- 既存 TT を入力とする TT-rounding
- tolerance による自動 rank 決定
- relative error による rank 決定
- TensorLy との照合
- TT 演算
- TT-matrix / MPO
- NN weight の TT 圧縮
- DMRG

今回は **dense tensor → TT-SVD → 固定 rank → 誤差上界** に限定します。


## 1. 単一 bond の最適性と複数 cut の違い

単一行列 $M$ の SVD を、

$$
M
=
U\Sigma V^T
=
\sum_{\alpha=1}^{\rho}
\sigma_\alpha u_\alpha v_\alpha^T
$$

とします。

rank を $r$ 以下に制限するなら、

$$
M_r
=
\sum_{\alpha=1}^{r}
\sigma_\alpha u_\alpha v_\alpha^T
$$

が EYM 定理による最良近似です。

$$
\boxed{
\|M-M_r\|_F^2
=
\sum_{\alpha=r+1}^{\rho}
\sigma_\alpha^2
}
$$

でした。

一方、$d$ 階テンソルでは cut が、

$$
1,\;2,\;\ldots,\;d-1
$$

と複数あります。

各 cut で truncated SVD を行うと、その1回の局所問題には EYM が使えます。

しかし1回目の truncation の後に扱うテンソルは、すでに元テンソルから変化しています。

したがって、

> 各局所 SVD がその時点で最良  
> $\not\Rightarrow$  
> 最後に得る TT が全 TT-rank 制約下で大域最良

です。

ここが Notebook 09・10 の単一 bond truncation と、今回の TT-SVD 誤差解析の違いです。


## 2. 元テンソルの各 cut と $\varepsilon_k$

テンソルを、

$$
A
\in
\mathbb{R}^{n_1\times n_2\times\cdots\times n_d}
$$

とします。

第 $k$ cut の unfolding を、

$$
A^{\langle k\rangle}
\in
\mathbb{R}^{
(n_1\cdots n_k)
\times
(n_{k+1}\cdots n_d)
}
$$

とします。

目標 TT-rank を、

$$
(r_1,r_2,\ldots,r_{d-1})
$$

とすると、今回使う $\varepsilon_k$ は、

$$
\boxed{
\varepsilon_k
:=
\min_{\operatorname{rank}(B)\le r_k}
\left\|
A^{\langle k\rangle}
-
B
\right\|_F
}
$$

です。

EYM 定理より、

$$
\boxed{
\varepsilon_k^2
=
\sum_{\ell>r_k}
\left(
\sigma_\ell
\left(
A^{\langle k\rangle}
\right)
\right)^2
}
$$

です。

### 重要

この Notebook では $\varepsilon_k$ を必ず、

> **元のテンソル $A$ の第 $k$ unfolding に対する最良 rank-$r_k$ 行列近似誤差**

という意味で使います。

TT-SVD の途中で現れる圧縮済み行列の局所 truncation error と、無条件に同一視しません。


## 3. 標準 TT-SVD の左→右構成

まず第1 unfolding を、

$$
A^{\langle1\rangle}
\in
\mathbb{R}^{n_1\times(n_2\cdots n_d)}
$$

とします。

SVD を行い、rank $r_1$ で truncation します。

$$
A^{\langle1\rangle}
=
U\Sigma V^T.
$$

上位 $r_1$ 成分を、

$$
U_1
=
U(:,1:r_1),
$$

$$
\Sigma_1
=
\Sigma(1:r_1,1:r_1),
$$

$$
V_1^T
=
V^T(1:r_1,:)
$$

とします。

第1 TT core は、

$$
G_1
=
\operatorname{reshape}(U_1)
\in
\mathbb{R}^{1\times n_1\times r_1}.
$$

残りを、

$$
\widehat A_1
=
\Sigma_1V_1^T
$$

とします。

次は $\widehat A_1$ を、

$$
(r_1n_2)
\times
(n_3\cdots n_d)
$$

へ reshape して再び SVD します。

同じ操作を左から右へ繰り返すことで、

$$
G_1,\;G_2,\;\ldots,\;G_d
$$

を構成します。


## 4. TT core の shape

標準 TT では、

$$
r_0=r_d=1
$$

とし、

$$
G_k
\in
\mathbb{R}^{r_{k-1}\times n_k\times r_k}
$$

です。

つまり、

$$
G_1
\in
\mathbb{R}^{1\times n_1\times r_1},
$$

$$
G_k
\in
\mathbb{R}^{r_{k-1}\times n_k\times r_k}
\qquad
(2\le k\le d-1),
$$

$$
G_d
\in
\mathbb{R}^{r_{d-1}\times n_d\times1}.
$$

再構成は、

$$
\widetilde A(i_1,\ldots,i_d)
=
\sum_{\alpha_1,\ldots,\alpha_{d-1}}
G_1(1,i_1,\alpha_1)
G_2(\alpha_1,i_2,\alpha_2)
\cdots
G_d(\alpha_{d-1},i_d,1)
$$

です。

rank 制限を入れなければ、各 SVD で必要な全成分を保持できるので、

$$
\boxed{
\|A-\widetilde A\|_F
\approx0
}
$$

になるはずです。


## 5. 誤差証明の最初の1ステップ

第1 unfolding の SVD truncation を、

$$
A^{\langle1\rangle}
=
U_1
\widehat A^{\langle1\rangle}
+
E_1^{\langle1\rangle}
$$

と書きます。

ここで、

- $U_1$：残した左特異ベクトル
- $\widehat A$：残した成分を次へ渡した縮約テンソル
- $E_1$：第1 cut で捨てた残差

です。

SVD の構造より、

$$
E_1^{\langle1\rangle}
=
U_-\Sigma_-V_-^T,
$$

かつ、

$$
U_1^TU_-=0.
$$

したがって $E_1$ は、残した左特異空間と直交する方向にあります。


## 6. 後続近似は残した部分空間の中にある

第1 cut 以降の TT-SVD によって、縮約テンソル $\widehat A$ を近似し、

$$
\widehat T
$$

を得たとします。

元の第1 mode へ埋め戻すと、

$$
T^{\langle1\rangle}
=
U_1
\widehat T^{\langle1\rangle}.
$$

また、第1 truncation 直後のテンソルを $B$ と書けば、

$$
B^{\langle1\rangle}
=
U_1
\widehat A^{\langle1\rangle}.
$$

よって、

$$
B^{\langle1\rangle}
-
T^{\langle1\rangle}
=
U_1
\left(
\widehat A^{\langle1\rangle}
-
\widehat T^{\langle1\rangle}
\right).
$$

つまり、後続ステップで生じる誤差 $B-T$ は、$U_1$ の列空間の中にあります。

一方、第1ステップの残差 $A-B=E_1$ は、その列空間と直交します。

したがって、

$$
\boxed{
\langle A-B,\;B-T\rangle_F
=
0
}
$$

です。


## 7. Pythagoras 型の誤差分解

$$
A-T
=
(A-B)
+
(B-T)
$$

であり、

$$
\langle A-B,\;B-T\rangle_F
=
0
$$

なので、

$$
\boxed{
\|A-T\|_F^2
=
\|A-B\|_F^2
+
\|B-T\|_F^2
}
$$

です。

第1 cut の truncation error は、

$$
\|A-B\|_F
=
\varepsilon_1
$$

なので、

$$
\|A-T\|_F^2
=
\varepsilon_1^2
+
\|B-T\|_F^2.
$$

さらに $U_1$ は列直交なので、

$$
U_1^TU_1=I
$$

であり、

$$
\|B-T\|_F
=
\|\widehat A-\widehat T\|_F.
$$

したがって、

$$
\boxed{
\|A-T\|_F^2
=
\varepsilon_1^2
+
\|\widehat A-\widehat T\|_F^2
}
$$

です。

これにより問題を1 mode 小さいテンソルへ移せます。


## 8. なぜ縮約後の残り cut の誤差は増えないか

### cut番号の記法

第1 mode を縮約した後のテンソル $\widehat A$ のモードは、

$$
(\alpha_1,i_2,\ldots,i_d)
$$

です。

したがって、元のテンソル $A$ の cut $k$ に対応する cut は、$\widehat A$ の内部では $k-1$ 番目の cut です。

この Notebook では、**元テンソル $A$ の cut 番号との対応を保つため**、

$$
\widehat\varepsilon_k
$$

という添字を使います。

つまり、

$$
k=2,\ldots,d-1
$$

に対し、$\widehat\varepsilon_k$ は

> 元の $A$ の cut $k$ に対応する、縮約後テンソル $\widehat A$ の cut $k-1$ における最良近似誤差

を表します。

---

$U_1$ は列直交なので、

$$
U_1^TU_1=I.
$$

したがって $U_1$ による埋め込みは Frobenius ノルムを保存します。

$$
\boxed{
\|U_1Z\|_F
=
\|Z\|_F
}
$$

一方、$U_1^T$ は元空間から残した部分空間へ縮約する写像です。

一般に、

$$
\boxed{
\|U_1^TY\|_F
\le
\|Y\|_F
}
$$

です。

ただし、この事実だけから直ちに

$$
\widehat\varepsilon_k
\le
\varepsilon_k
$$

が出るわけではありません。

間に、

> 元テンソルの最良 rank-$r_k$ 近似から、縮約後テンソルの rank-$r_k$ 近似候補を1つ作る

という一手が必要です。

元テンソル $A$ の cut $k$ について、最良 rank-$r_k$ 近似を

$$
A^{\langle k\rangle}
=
B_k
+
E_k,
$$

$$
\operatorname{rank}(B_k)
\le
r_k,
\qquad
\|E_k\|_F
=
\varepsilon_k
$$

とします。

第1 mode を $U_1^T$ で縮約した、対応する unfolding では、

$$
\widehat A^{\langle k-1\rangle}
=
U_1^T
A^{\langle k\rangle}.
$$

したがって、

$$
\widehat A^{\langle k-1\rangle}
=
U_1^TB_k
+
U_1^TE_k
$$

と書けます。

ここで、左から行列を掛けても rank は増えないので、

$$
\operatorname{rank}(U_1^TB_k)
\le
\operatorname{rank}(B_k)
\le
r_k.
$$

よって、

$$
U_1^TB_k
$$

は、

$$
\widehat A^{\langle k-1\rangle}
$$

に対する rank-$r_k$ 近似の **許される候補**です。

縮約後テンソルの最良 rank-$r_k$ 近似誤差を $\widehat\varepsilon_k$ とすると、

$$
\widehat\varepsilon_k
=
\min_{\operatorname{rank}(\widehat B)\le r_k}
\left\|
\widehat A^{\langle k-1\rangle}
-
\widehat B
\right\|_F.
$$

最良誤差は任意の許される候補の誤差以下なので、

$$
\begin{aligned}
\widehat\varepsilon_k
&\le
\left\|
\widehat A^{\langle k-1\rangle}
-
U_1^TB_k
\right\|_F
\\
&=
\|U_1^TE_k\|_F
\\
&\le
\|E_k\|_F
\\
&=
\varepsilon_k.
\end{aligned}
$$

したがって、

$$
\boxed{
\widehat\varepsilon_k
\le
\varepsilon_k
}
\qquad
(k=2,\ldots,d-1)
$$

です。

ここで使った不等号は2種類あります。

1. 最初の不等号

   $$
   \widehat\varepsilon_k
   \le
   \left\|
   \widehat A^{\langle k-1\rangle}
   -
   U_1^TB_k
   \right\|_F
   $$

   は、

   > **最良誤差 $\le$ 許される候補の誤差**

   という最適化問題の定義から来ます。

2. 最後の不等号

   $$
   \|U_1^TE_k\|_F
   \le
   \|E_k\|_F
   $$

   は、

   > $U_1^T$ が列直交部分空間への縮約であり、Frobenius ノルムを増やさない

   ことから来ます。

この2段階を分けて考えることで、

$$
\widehat\varepsilon_k
\le
\varepsilon_k
$$

の意味が明確になります。


## 9. 帰納法による TT-SVD の誤差上界

第1ステップでは、

$$
\|A-T\|_F^2
=
\varepsilon_1^2
+
\|\widehat A-\widehat T\|_F^2
$$

でした。

残りの $(d-1)$ 階テンソル $\widehat A$ に同じ議論を適用すると、

$$
\|\widehat A-\widehat T\|_F^2
\le
\sum_{k=2}^{d-1}
\widehat\varepsilon_k^2.
$$

さらに、

$$
\widehat\varepsilon_k
\le
\varepsilon_k
$$

なので、

$$
\|\widehat A-\widehat T\|_F^2
\le
\sum_{k=2}^{d-1}
\varepsilon_k^2.
$$

したがって、

$$
\boxed{
\|A-T\|_F^2
\le
\sum_{k=1}^{d-1}
\varepsilon_k^2
}
$$

を得ます。

よって、

$$
\boxed{
\|A-T\|_F
\le
\left(
\sum_{k=1}^{d-1}
\varepsilon_k^2
\right)^{1/2}
}
$$

です。


## 10. 大域最適 TT 近似との比較

指定 TT-rank、

$$
(r_1,\ldots,r_{d-1})
$$

以下での大域最適近似を、

$$
A_\ast
$$

とし、

$$
e_\ast
=
\|A-A_\ast\|_F
$$

とします。

$A_\ast$ は各 cut で、

$$
\operatorname{rank}
\left(
A_\ast^{\langle k\rangle}
\right)
\le
r_k
$$

を満たします。

したがって $A_\ast^{\langle k\rangle}$ は、$\varepsilon_k$ を定義する行列近似問題の候補です。

よって、

$$
\begin{aligned}
\varepsilon_k
&=
\min_{\operatorname{rank}(B)\le r_k}
\left\|
A^{\langle k\rangle}-B
\right\|_F
\\
&\le
\left\|
A^{\langle k\rangle}
-
A_\ast^{\langle k\rangle}
\right\|_F
\\
&=
\|A-A_\ast\|_F
\\
&=
e_\ast.
\end{aligned}
$$

したがって、

$$
\varepsilon_k
\le
e_\ast
$$

です。


## 11. TT-SVD の準最適性

前節までの結果を使うと、

$$
\begin{aligned}
\|A-T\|_F^2
&\le
\sum_{k=1}^{d-1}
\varepsilon_k^2
\\
&\le
\sum_{k=1}^{d-1}
e_\ast^2
\\
&=
(d-1)e_\ast^2.
\end{aligned}
$$

したがって、

$$
\boxed{
\|A-T\|_F
\le
\sqrt{d-1}
\,
e_\ast
}
$$

すなわち、

$$
\boxed{
\|A-T\|_F
\le
\sqrt{d-1}
\,
\|A-A_\ast\|_F
}
$$

です。

### ここで主張していないこと

TT-SVD が、

$$
T=A_\ast
$$

になるとは主張していません。

また、

> TT-SVD で得られる TT は、任意の別構成 TT より必ず誤差が小さい

とも主張していません。

主張は、

> **TT-SVD の誤差は、大域最適 TT 近似誤差の $\sqrt{d-1}$ 倍以内に保証される**

という準最適性です。


## 12. 今回の数値検証用セットアップ

まず warm-up 用に、複数の特異値を捨てる行列を作ります。

特異値を、

$$
(5,\;2,\;0.5,\;0.1)
$$

とし、rank $r=2$ に truncation する例です。

この場合、理論上の二乗誤差は、

$$
0.5^2+0.1^2
=
0.26
$$

です。

その後、TT-SVD 用に、

$$
A\in\mathbb{R}^{4\times4\times4\times4}
$$

の小さいランダムテンソルを用意します。

固定 rank の実験では、例えば、

$$
(r_1,r_2,r_3)
=
(2,4,2)
$$

を使います。

この rank は今回の数値実験用の固定値であり、自動選択ではありません。


In [12]:
import torch

torch.set_default_dtype(torch.float64)
torch.manual_seed(11)

# --- Exercise 1 用：特異値を制御した行列 ---
m, n = 6, 4

Q_left, _ = torch.linalg.qr(
    torch.randn(m, n),
    mode="reduced",
)
Q_right, _ = torch.linalg.qr(
    torch.randn(n, n),
    mode="reduced",
)

matrix_singular_values = torch.tensor(
    [5.0, 2.0, 0.5, 0.1]
)

M = (
    Q_left
    @ torch.diag(matrix_singular_values)
    @ Q_right.T
)

matrix_keep_rank = 2


# --- Exercise 2, 3 用：4階 dense tensor ---
A = torch.randn(4, 4, 4, 4)

tensor_shape = tuple(A.shape)
d = A.ndim

# 固定 rank TT-SVD 用
target_tt_ranks = [2, 4, 2]

print("M shape:", tuple(M.shape))
print("target singular values:", matrix_singular_values)
print("A shape:", tensor_shape)
print("target TT ranks:", target_tt_ranks)


M shape: (6, 4)
target singular values: tensor([5.0000, 2.0000, 0.5000, 0.1000])
A shape: (4, 4, 4, 4)
target TT ranks: [2, 4, 2]


## 13. 演習1 — 単一行列の Truncated SVD を再確認する

### 目的

今回のTT-SVD誤差解析の最小単位である、

$$
\|M-M_r\|_F^2
=
\sum_{\ell>r}\sigma_\ell^2
$$

を数値確認します。

Notebook 09 では1個の特異値だけを捨てました。

今回は、

$$
(\sigma_1,\sigma_2,\sigma_3,\sigma_4)
=
(5,2,0.5,0.1)
$$

から、

$$
r=2
$$

として **2個の特異値を捨てます**。

### TODO

1. `M` に reduced SVD を適用する
2. rank `matrix_keep_rank` の truncated SVD を構成する
3. 
   $$
   \|M-M_r\|_F^2
   $$
   を計算する
4. 捨てた特異値について
   $$
   \sum_{\ell>r}\sigma_\ell^2
   $$
   を計算する
5. 両者の差を確認する

### 考えること

- 今回は捨てる特異値が1個ではなく2個である
- Frobenius誤差は $\sigma_{r+1}$ だけではなく、捨てた全成分の二乗和になる
- この等式は各TT-SVDステップで使う局所SVDの基本単位である


In [13]:
# TODO 1:
# 単一行列の truncated SVD error を確認してください。
from nn_compression.compression.svd import truncated_svd, matrix_max_rank
#
# 1. M に reduced SVD
U, S, Vh = torch.linalg.svd(M, full_matrices=False)
#
# 2. rank-r の M_r を構成
rank = matrix_max_rank(M)
r=2
assert 1 <= r <= rank
M_r = U[:, :r] @ torch.diag(S[:r]) @ Vh[:r, :]  # r=2 なら先頭2本だけ
#
# 3. ||M - M_r||_F^2
reconstrunction_error=(torch.norm(M-M_r))**2
#
# 4. discarded singular values の二乗和
discarded_sq = torch.linalg.vector_norm(S[r:]).square()
# 5. 両者の差を表示
error_gap = abs(reconstrunction_error.item() - discarded_sq.item())
print("||M - M_r||_F^2 =", reconstrunction_error.item())
print("sum_{l>r} sigma^2 =", discarded_sq.item())
print("difference        =", error_gap)


||M - M_r||_F^2 = 0.2599999999999998
sum_{l>r} sigma^2 = 0.2599999999999999
difference        = 1.1102230246251565e-16


## 14. 演習2 — Rank 制限なしの 4階 TT-SVD

### 目的

まず truncation を入れずに TT-SVD を実装し、

$$
A
\approx
G_1G_2G_3G_4
$$

を丸め誤差水準で再構成できることを確認します。

入力は、

$$
A
\in
\mathbb{R}^{4\times4\times4\times4}
$$

です。

完全 rank の場合、各ステップで SVD の全成分を保持します。

### TODO

1. 第1 mode から左→右に TT-SVD を行う
2. 各ステップで SVD する行列の shape を記録する
3. 各ステップの保持 rank を記録する
4. 各 `U` を
   $$
   (r_{k-1},n_k,r_k)
   $$
   へ reshape して TT core にする
5. 最後の残差行列を最終 core にする
6. TT core を順に縮約して `A_exact_tt` を再構成する
7. 
   $$
   \|A-A_{\mathrm{exact\ TT}}\|_F
   $$
   を確認する

### 考えること

- 第1ステップの行列 shape は何か
- 第2ステップでは、なぜ行数に直前の TT-rank が入るのか
- exact TT-SVD の bond rank と、元テンソルの unfolding rank はどう対応するか
- truncation をしていないので、なぜ理論上は情報を失わないのか


In [14]:
# TODO 2:
# rank 制限なしの TT-SVD を実装してください。
from nn_compression.compression import tt_svd_exact, tt_reconstruct
# 記録したいもの:
# - 各 step の matrix shape
# - 各 step の singular values
# - 各 bond rank
# - 各 TT core shape
#
# 1. A から左→右に exact TT-SVD
#    各SVDは torch.linalg.svd(..., full_matrices=False) を使う
# 2. TT cores を作る
G1,G2,G3,G4=tt_svd_exact(A)

# 3. cores を縮約して A_exact_tt を再構成
A_exact_tt = tt_reconstruct([G1, G2, G3, G4])
# 4. reconstruction error を表示
reconstruction_error=torch.linalg.vector_norm(A_exact_tt-A)
print("reconstruction_error:",reconstruction_error.item())


reconstruction_error: 3.4675948148250965e-14


## 15. 固定 Rank TT-SVD で何を比較するか

次は、

$$
(r_1,r_2,r_3)
=
(2,4,2)
$$

へ固定して TT-SVD を行います。

最終的に得る近似テンソルを、

$$
T
$$

とします。

ここで確認する中心式は、

$$
\boxed{
\|A-T\|_F
\le
\left(
\varepsilon_1^2
+
\varepsilon_2^2
+
\varepsilon_3^2
\right)^{1/2}
}
$$

です。

ただし、

$$
\varepsilon_k
$$

は **TT-SVD途中の局所SVDで捨てた量ではありません**。

各 $\varepsilon_k$ は元の $A$ から直接作った unfolding、

$$
A^{\langle k\rangle}
$$

の特異値から、

$$
\varepsilon_k^2
=
\sum_{\ell>r_k}
\sigma_\ell
\left(
A^{\langle k\rangle}
\right)^2
$$

として別に計算します。


## 16. 元テンソルの3つの unfolding

今回、

$$
A
\in
\mathbb{R}^{4\times4\times4\times4}
$$

なので、

### cut 1

$$
A^{\langle1\rangle}
\in
\mathbb{R}^{4\times64}
$$

### cut 2

$$
A^{\langle2\rangle}
\in
\mathbb{R}^{16\times16}
$$

### cut 3

$$
A^{\langle3\rangle}
\in
\mathbb{R}^{64\times4}
$$

です。

目標 rank が、

$$
(r_1,r_2,r_3)
=
(2,4,2)
$$

なら、

$$
\varepsilon_1^2
=
\sum_{\ell>2}
\sigma_\ell
\left(
A^{\langle1\rangle}
\right)^2,
$$

$$
\varepsilon_2^2
=
\sum_{\ell>4}
\sigma_\ell
\left(
A^{\langle2\rangle}
\right)^2,
$$

$$
\varepsilon_3^2
=
\sum_{\ell>2}
\sigma_\ell
\left(
A^{\langle3\rangle}
\right)^2.
$$

これらは **すべて元テンソル $A$ の unfolding** から求めます。


## 17. 演習3 — 固定 Rank TT-SVD と準最適誤差上界

### 目的

固定 TT-rank、

$$
(2,4,2)
$$

で TT-SVD を実行し、実測誤差と、

$$
\sqrt{
\varepsilon_1^2
+
\varepsilon_2^2
+
\varepsilon_3^2
}
$$

を比較します。

### TODO A：固定 rank TT-SVD

1. `target_tt_ranks` を各cutの保持rankとして使う
2. 左→右 TT-SVD を実行する
3. 各ステップで truncated SVD を行う
4. TT core を保存する
5. round 後ではなく、**dense tensor から新規構築する TT-SVD** であることを確認する
6. TTを再構成して `A_tt` を作る
7. 実測誤差
   $$
   \|A-A_{\mathrm{tt}}\|_F
   $$
   を計算する

### TODO B：元の unfolding から $\varepsilon_k$ を求める

1. 元の `A` から cut 1, 2, 3 の unfolding をそれぞれ作る
2. 各 unfolding に reduced SVD を適用する
3. 各目標 rank より後ろの特異値の二乗和を求める
4. 
   $$
   \varepsilon_1,\varepsilon_2,\varepsilon_3
   $$
   を求める
5. 
   $$
   \text{bound}
   =
   \sqrt{
   \sum_k\varepsilon_k^2
   }
   $$
   を求める

### TODO C：不等式を確認する

$$
\boxed{
\|A-A_{\mathrm{tt}}\|_F
\le
\text{bound}
}
$$

を float64 の許容誤差を考慮して確認してください。

### 考えること

- 実測誤差と上界は等号になる必要があるか
- TT-SVD途中の局所 discarded weight と、ここで定義した $\varepsilon_k$ を同一視してよいか
- 上界が実測誤差より大きくなることは失敗を意味するか
- この実験だけで大域最適 TT $A_\ast$ を求めたことになるか


In [15]:
# TODO 3:
# 固定 rank TT-SVD と誤差上界を確認してください。
from nn_compression.compression import tt_svd_ranks,tt_unfold
# Part A:
target_tt_ranks = [2, 4, 2]
# - dense A から左→右に TT-SVD
# - 各SVDは torch.linalg.svd(..., full_matrices=False) を使う
# - A_tt を再構成
# - actual_error = ||A - A_tt||_Ftt
G1,G2,G3,G4=tt_svd_ranks(A,target_tt_ranks)
A_tt=tt_reconstruct([G1,G2,G3,G4])
actual_error=torch.linalg.vector_norm(A_tt-A)
print("reconstruction_actual_error:",actual_error.item())

# Part B:
# - 元の A の cut 1, 2, 3 unfolding を作る
# - 各 unfolding の singular values を求める
# - epsilon_k^2 = tail singular-value energy
# - bound = sqrt(sum epsilon_k^2)
# Part B:
# 元テンソル A の各 cut unfolding から ε_k を求める。
# ε_k^2 = sum_{ℓ > r_k} σ_ℓ(A^{<k>})^2 （EYM の最良 rank-r_k 誤差）
# bound = sqrt(sum_k ε_k^2)
epsilon_sq = []
for k, r_k in enumerate(target_tt_ranks, start=1):
    # cut k の unfolding A^{<k>} の特異値
    S_k = torch.linalg.svdvals(tt_unfold(A, k))
    # 目標 rank r_k より後ろ（捨てる側）の二乗和 = ε_k^2
    eps_k_sq = S_k[r_k:].square().sum()
    epsilon_sq.append(eps_k_sq)
    print(f"epsilon_{k}^2 =", eps_k_sq.item())
epsilon_sq = torch.stack(epsilon_sq)
bound = epsilon_sq.sum().sqrt()  # ||A-T||_F の理論上界
print("bound =", bound.item())
# Part C:
# actual_error <= bound を確認
# （float64 の丸めを見込んで小さな eps を許容）
eps = 1e-12
print("actual_error =", actual_error.item())
print("bound        =", bound.item())
print("actual_error <= bound + eps ?", actual_error.item() <= bound.item() + eps)
#
# 注意:
# TT-SVD途中の局所 discarded error を epsilon_k と
# 無条件に同一視しないこと。



reconstruction_actual_error: 13.627066726245229
epsilon_1^2 = 112.02491601313216
epsilon_2^2 = 115.50992877359565
epsilon_3^2 = 102.07805194389165
bound = 18.15524433133907
actual_error = 13.627066726245229
bound        = 18.15524433133907
actual_error <= bound + eps ? True


## 18. なぜ $U$ と $U^T$ のノルム挙動を分けるのか

$U\in\mathbb{R}^{m\times r}$ が列直交、

$$
U^TU=I_r
$$

とします。

### 埋め込み

任意の、

$$
Z\in\mathbb{R}^{r\times p}
$$

について、

$$
\begin{aligned}
\|UZ\|_F^2
&=
\operatorname{tr}
\left(
Z^TU^TUZ
\right)
\\
&=
\operatorname{tr}(Z^TZ)
\\
&=
\|Z\|_F^2.
\end{aligned}
$$

よって、

$$
\boxed{
\|UZ\|_F
=
\|Z\|_F
}
$$

です。

### 縮約

一方、

$$
Y\in\mathbb{R}^{m\times p}
$$

について、

$$
U^TY
$$

は $Y$ を $\operatorname{col}(U)$ へ縮約します。

一般に、

$$
\boxed{
\|U^TY\|_F
\le
\|Y\|_F
}
$$

です。

等号は常に成立するわけではありません。

この違いが、

- 縮約側では誤差が増えない
- 埋め戻し側では誤差ノルムをそのまま保存する

という TT-SVD の帰納的誤差評価に使われます。


## 19. 演習4 — $U$ と $U^T$ の Frobenius ノルム実験

### 目的

列直交行列 $U$ をランダムに作り、

$$
\|UZ\|_F
=
\|Z\|_F
$$

と、

$$
\|U^TY\|_F
\le
\|Y\|_F
$$

を数値確認します。

### TODO

1. 縦長のランダム行列を reduced QR して、列直交行列 `U_iso` を作る
2. 
   $$
   U_{\mathrm{iso}}^TU_{\mathrm{iso}}
   \approx
   I
   $$
   を確認する
3. shape が適合するランダム `Z` を作る
4. 
   $$
   \|U_{\mathrm{iso}}Z\|_F
   $$
   と
   $$
   \|Z\|_F
   $$
   を比較する
5. shape が適合するランダム `Y` を作る
6. 
   $$
   \|U_{\mathrm{iso}}^TY\|_F
   $$
   と
   $$
   \|Y\|_F
   $$
   を比較する

### 考えること

- なぜ $UZ$ は等号なのに $U^TY$ は不等号なのか
- $UU^T$ は何を表すか
- `Y` が完全に $\operatorname{col}(U)$ に入っている場合はどうなるか
- この性質が $\widehat\varepsilon_k\le\varepsilon_k$ とどうつながるか


In [16]:
# TODO 4:
# 列直交 U による埋め込みと縮約のノルム挙動を確認してください。
#
# 1. reduced QR で U_iso を作る
A_tall = torch.randn(20, 5)
Q, R = torch.linalg.qr(A_tall, mode="reduced")
U_iso=Q
# 2. U_iso.T @ U_iso ≈ I を確認
U_eye=U_iso.T@U_iso
I = torch.eye(U_eye.shape[1], dtype=U_eye.dtype, device=U_eye.device)
eye_error=torch.norm(U_eye - I)
# 3. Z を作り、||U_iso @ Z||_F と ||Z||_F を比較
#    U_iso: (20, 5) なので Z は (5, *)
Z = torch.randn(U_iso.shape[1], 3)
UZ_norm = torch.linalg.vector_norm(U_iso @ Z).item()
Z_norm = torch.linalg.vector_norm(Z).item()
print("||U_iso|| gram error =", eye_error.item())
print("||U_iso @ Z||_F =", UZ_norm)
print("||Z||_F         =", Z_norm)
print("||UZ|| - ||Z||  =", abs(UZ_norm - Z_norm))

# 4. Y を作り、||U_iso.T @ Y||_F と ||Y||_F を比較
#    U_iso.T: (5, 20) なので Y は (20, *)
Y = torch.randn(U_iso.shape[0], 3)
UTY_norm = torch.linalg.vector_norm(U_iso.T @ Y).item()
Y_norm = torch.linalg.vector_norm(Y).item()
print("||U_iso.T @ Y||_F =", UTY_norm)
print("||Y||_F           =", Y_norm)
print("||Y|| - ||U^T Y|| =", Y_norm - UTY_norm)  # >= 0 のはず

||U_iso|| gram error = 6.346208282740924e-16
||U_iso @ Z||_F = 6.118831066242175
||Z||_F         = 6.1188310662421745
||UZ|| - ||Z||  = 8.881784197001252e-16
||U_iso.T @ Y||_F = 3.6355592483231116
||Y||_F           = 6.56559073251899
||Y|| - ||U^T Y|| = 2.930031484195878


## 20. 今回の理論で区別する3種類の誤差

今回、似た誤差量が複数出てきます。

混同しないように整理します。

### 1. 単一行列 SVD の truncation error

$$
\|M-M_r\|_F^2
=
\sum_{\ell>r}\sigma_\ell(M)^2.
$$

これは EYM により、その行列に対する最良 rank-$r$ 誤差です。

### 2. 元テンソルの各 unfolding の最良 rank 誤差

$$
\varepsilon_k
=
\min_{\operatorname{rank}(B)\le r_k}
\|A^{\langle k\rangle}-B\|_F.
$$

TT-SVD の準最適誤差上界に使う $\varepsilon_k$ はこれです。

### 3. TT-SVD の最終実測誤差

$$
\|A-T\|_F.
$$

保証されるのは、

$$
\boxed{
\|A-T\|_F^2
\le
\sum_{k=1}^{d-1}
\varepsilon_k^2
}
$$

です。

これらを同じ記号・同じ意味として扱わないことが重要です。


## 21. TT-SVD と TT-rounding の違い

今回扱うのは **TT-SVD** です。

入力は dense tensor、

$$
A
\in
\mathbb{R}^{n_1\times\cdots\times n_d}
$$

です。

そこから左→右に SVD を繰り返し、TT core を新規構築します。

一方、次に学ぶ **TT-rounding** は、

> すでに TT core として与えられているテンソルの rank をさらに縮める

操作です。

典型的には、

1. QR による正準化
2. SVD truncation
3. 隣接 core への吸収

を使います。

今回の準最適性証明で使った直交性・等長性・SVD truncation は TT-rounding でも重要ですが、**アルゴリズムの入力と操作順は別物**です。


## 22. 今回のまとめ

### 単一行列

EYM 定理より、

$$
\boxed{
\|M-M_r\|_F^2
=
\sum_{\ell>r}
\sigma_\ell^2
}
$$

です。

### 標準 TT-SVD

元テンソルの各 cut に対して、

$$
\varepsilon_k
=
\min_{\operatorname{rank}(B)\le r_k}
\|A^{\langle k\rangle}-B\|_F
$$

とすると、

$$
\boxed{
\|A-T\|_F^2
\le
\sum_{k=1}^{d-1}
\varepsilon_k^2
}
$$

です。

### 大域最適 TT との比較

指定 rank 以下の大域最適近似を $A_\ast$ とすると、

$$
\varepsilon_k
\le
\|A-A_\ast\|_F
$$

なので、

$$
\boxed{
\|A-T\|_F
\le
\sqrt{d-1}
\,
\|A-A_\ast\|_F
}
$$

です。

### 今回の数値確認

1. 複数特異値を捨てる単一行列 SVD
2. rank 制限なし TT-SVD の完全再構成
3. 固定 rank TT-SVD と
   $$
   \sqrt{\sum_k\varepsilon_k^2}
   $$
   の比較
4. 
   $$
   \|UZ\|_F=\|Z\|_F,
   \qquad
   \|U^TY\|_F\le\|Y\|_F
   $$
   の確認

までを扱います。

次は、既存の TT core を入力として rank を落とす **TT-rounding** に進みます。
